# Part II — Explanatory Visualizations: Ford GoBike (Feb 2019)

This notebook produces polished explanatory figures for presentation, using the cleaned results from Part I. Run the cells to generate and save high-resolution figures for slides.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
# Load data (same CSV in repo root)
df = pd.read_csv('201902-fordgobike-tripdata.csv')
df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
df['duration_min'] = df['duration_sec'] / 60.0
df['member_birth_year'] = pd.to_numeric(df['member_birth_year'], errors='coerce')
df['age'] = 2019 - df['member_birth_year']
df['start_hour'] = df['start_time'].dt.hour
df['start_day'] = df['start_time'].dt.day_name()
df['is_weekend'] = df['start_day'].isin(['Saturday','Sunday'])
# Clean subset for plotting
clean = df[(df['duration_min']>0) & (df['age'].between(10,100))].copy()
clean['period'] = clean['start_hour'].apply(lambda h: 'MorningCommute' if 5<=h<9 else ('EveningCommute' if 15<=h<19 else ('Midday' if 9<=h<15 else ('LateNight' if h<5 else 'Night'))))

## 1) Commute heatmap — counts by weekday and hour

In [ ]:
pivot = clean.pivot_table(index='start_day', columns='start_hour', values='duration_sec', aggfunc='count').fillna(0)
days_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
pivot = pivot.reindex(days_order)
plt.figure(figsize=(14,4))
ax = sns.heatmap(pivot, cmap='YlOrRd')
ax.set_xlabel('Start hour')
ax.set_ylabel('Weekday')
ax.set_title('Commute heatmap — trip counts by weekday and hour')
plt.savefig('none/commute_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## 2) Top stations — bar chart with callouts

In [ ]:
top10 = clean['start_station_name'].value_counts().nlargest(10)
plt.figure(figsize=(8,5))
ax = sns.barplot(y=top10.index, x=top10.values, palette='viridis')
ax.set_xlabel('Trip starts')
ax.set_title('Top 10 start stations')
# annotate top three
for i, v in enumerate(top10.values):
    ax.text(v + max(top10.values)*0.01, i, str(v), va='center')
plt.savefig('none/top10_start_stations.png', dpi=200, bbox_inches='tight')
plt.show()

## 3) Duration comparison — Subscribers vs Customers (annotated)

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=clean[clean['duration_min']<=240], x='user_type', y='duration_min', palette=['#4c72b0','#dd8452'])
plt.ylim(0,120)
plt.xlabel('User type')
plt.ylabel('Duration (min)')
plt.title('Trip duration comparison: Subscribers vs Customers (truncated)')
# add medians text
meds = clean.groupby('user_type')['duration_min'].median()
for i,u in enumerate(meds.index):
    plt.text(i, meds.loc[u]+3, f
, ha='center')
plt.savefig('none/duration_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

## Notes
- Run this notebook after running the Part I exploration notebook to ensure any saved cleaned datasets or images exist.
- The three PNGs are saved inside the `none` folder for easy inclusion in slide decks.